# Support Vector Machine Classification

## Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
dataset = fetch_ucirepo(id=267)
X = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].copy()
print('Dataset shape:', X.shape)
print('Missing values:', X.isnull().sum().sum())
print('Features:', X.columns.tolist())
print('Class distribution:')
print(y.value_counts())

## Exploratory Data Analysis

In [ ]:
display(X.head())
display(X.describe())

In [ ]:
X.hist(figsize=(10, 7), bins=25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.tight_layout()
plt.show()

## Data Preparation

In [ ]:
print('Constant features:', X.columns[X.nunique() <= 1].tolist())
duplicate_pairs = []
for i in range(X.shape[1]):
    for j in range(i + 1, X.shape[1]):
        if X.iloc[:, i].equals(X.iloc[:, j]):
            duplicate_pairs.append((X.columns[i], X.columns[j]))
print('Exact duplicate feature pairs:', duplicate_pairs)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## Dimensionality Reduction

In [ ]:
pca_full = PCA(n_components=4)
X_train_full = pca_full.fit_transform(X_train_scaled)
X_test_full = pca_full.transform(X_test_scaled)
pca_reduced = PCA(n_components=2)
X_train_reduced = pca_reduced.fit_transform(X_train_scaled)
X_test_reduced = pca_reduced.transform(X_test_scaled)
print('Full PCA dimensions:', X_train_full.shape[1])
print('Full PCA variance retained:', pca_full.explained_variance_ratio_.sum())
print('Reduced PCA dimensions:', X_train_reduced.shape[1])
print('Reduced PCA variance retained:', pca_reduced.explained_variance_ratio_.sum())

## SVM Model Training

In [ ]:
svm_full = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_full.fit(X_train_full, y_train)
pred_full = svm_full.predict(X_test_full)
svm_reduced = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_reduced.fit(X_train_reduced, y_train)
pred_reduced = svm_reduced.predict(X_test_reduced)

## Model Evaluation

In [ ]:
print('SVM with four PCA components')
print('Accuracy:', accuracy_score(y_test, pred_full))
print(classification_report(y_test, pred_full))
print('SVM with two PCA components')
print('Accuracy:', accuracy_score(y_test, pred_reduced))
print(classification_report(y_test, pred_reduced))

In [ ]:
cm = confusion_matrix(y_test, pred_full)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['SVM - 4 PCA components', 'SVM - 2 PCA components'],
    'Dimensions': [4, 2],
    'Variance Retained': [
        pca_full.explained_variance_ratio_.sum(),
        pca_reduced.explained_variance_ratio_.sum()
    ],
    'Test Accuracy': [
        accuracy_score(y_test, pred_full),
        accuracy_score(y_test, pred_reduced)
    ]
})
display(results)

In [ ]:
full_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=4)),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale'))
])
cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='accuracy')
print('Cross-validation scores:', cv_scores)
print('Mean cross-validation accuracy:', cv_scores.mean())